In [1]:
!pip install -q torch transformers datasets tokenizers \
    sentence-transformers rank-bm25 pytrec-eval-terrier tqdm scipy comet-ml logging

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.2/786.2 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.2 MB/s eta 0:00:00


In [2]:
from diploma_code import build_model_and_tokenizer
from diploma_code import ColBERTIndexer, ColBERTRetriever
from diploma_code import ModelConfig, IndexConfig

In [3]:
device = "cuda"

In [4]:
model_cfg = ModelConfig(encoder_name="xlm-roberta-base", embedding_dim=128)
model, tokenizer = build_model_and_tokenizer(model_cfg)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
import torch

ckpt = torch.load("/kaggle/input/models/vikakuznetzowa/diploma-phase2/pytorch/default/1/checkpoints/checkpoint_phase2_best.pt", map_location="cpu")
model.load_state_dict(ckpt["model_state_dict"])

<All keys matched successfully>

In [ ]:
import json
with open("/kaggle/input/datasets/vikakuznetzowa/diploma-ru-hard-negatives/corpus.jsonl") as f:
    corpus = [json.loads(line) for line in f]

In [ ]:
index_cfg = IndexConfig(
    index_dir="/kaggle/working/index_finetuned",
    batch_size=256,
    save_fp16=True,
)
indexer = ColBERTIndexer(model, tokenizer, index_cfg, model_cfg)
indexer.build(
    texts=[d["text"] for d in corpus],
    doc_ids=[d["id"] for d in corpus],
)

Indexing: 100%|██████████| 137/137 [05:51<00:00,  2.56s/it]


PosixPath('/kaggle/working/index_finetuned')

In [ ]:
retriever = ColBERTRetriever(
    model, tokenizer,
    index_dir=index_cfg.index_dir,
    model_cfg=model_cfg,
    device=device,
)

with open("/kaggle/input/datasets/vikakuznetzowa/diploma-ru-hard-negatives/queries.jsonl") as f:
    queries = [json.loads(line) for line in f]

In [9]:
import os
from tqdm import tqdm
ft_results = {}
for q in tqdm(queries, desc="Retrieval progress"):
    hits = retriever.retrieve(q["text"], top_k=100)
    ft_results[q["id"]] = hits

ft_path = "/kaggle/working/results/finetuned_results.json"
os.makedirs(os.path.dirname(ft_path), exist_ok=True)
with open(ft_path, "w") as f:
    json.dump({"ColBERT-RU_finetuned": ft_results}, f, ensure_ascii=False)

Retrieval progress: 100%|██████████| 1252/1252 [58:06<00:00,  2.78s/it]
